In [47]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
import os
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Paths
SILVER_PATH = "../data/silver/"
GOLD_PATH = "../data/gold/"

if not os.path.exists(GOLD_PATH):
    os.makedirs(GOLD_PATH)

print("="*60)
print("GOLD LAYER - Star Schema Creation")
print("="*60)

# Load silver data (AS IS - no renaming)
cert_silver = pd.read_csv(os.path.join(SILVER_PATH, "certifications_silver.csv"))
clubs_silver = pd.read_csv(os.path.join(SILVER_PATH, "clubs_silver.csv"))
results_silver = pd.read_csv(os.path.join(SILVER_PATH, "results_silver.csv"))

print(f"Certifications shape: {cert_silver.shape}")
print(f"Clubs shape: {clubs_silver.shape}")
print(f"Results shape: {results_silver.shape}")

print("\n" + "="*60)
print("CERTIFICATIONS COLUMNS:")
print(cert_silver.columns.tolist())

print("\nCLUBS COLUMNS:")
print(clubs_silver.columns.tolist())

print("\nRESULTS COLUMNS:")
print(results_silver.columns.tolist())

GOLD LAYER - Star Schema Creation
Certifications shape: (20221, 10)
Clubs shape: (436, 15)
Results shape: (112296, 12)

CERTIFICATIONS COLUMNS:
['Code', 'person_type_clean', 'gender_clean', 'calculated_age', 'dob_missing', 'mental_handicap_flag', 'parents_consent_flag', 'hap_flag', 'unified_partner_flag', 'bronze_timestamp']

CLUBS COLUMNS:
['club_id', 'club_name', 'region', 'Country', 'total_participations', 'bronze_timestamp', 'Participation Games 2015', 'Participation Games 2016', 'Participation Games 2017', 'Participation Games 2018', 'Participation Games 2019', 'Participation Games 2022', 'Participation Games 2023', 'Participation Games 2024', 'Participation Games 2025']

RESULTS COLUMNS:
['athlete_id', 'club_name', 'gender', 'age', 'sport_name', 'year', 'score', 'rank', 'is_disqualified', 'rank_missing_flag', 'score_missing_flag', 'bronze_timestamp']


In [ ]:
# Cell 2: Create dim_athlete from results (using original Code from bronze)
print("="*60)
print("Creating dim_athlete from results data")
print("="*60)

# Extract unique athletes from results_silver with their demographics
dim_athlete = results_silver[[
    'athlete_id', 'gender', 'age'
]].drop_duplicates(subset=['athlete_id'], keep='first').copy()

# Rename for clarity
dim_athlete = dim_athlete.rename(columns={
    'athlete_id': 'code',
    'gender': 'gender',
    'age': 'age'
})

# Calculate years of participation for each athlete
athlete_years = results_silver.groupby('athlete_id')['year'].agg(['min', 'max', 'nunique']).reset_index()
athlete_years = athlete_years.rename(columns={
    'athlete_id': 'code',
    'min': 'first_year',
    'max': 'last_year',
    'nunique': 'years_participated'
})

dim_athlete = dim_athlete.merge(athlete_years, on='code', how='left')

# Calculate experience (number of years since first year)
dim_athlete['experience_years'] = dim_athlete['last_year'] - dim_athlete['first_year'] + 1

# Add timestamp
dim_athlete['gold_created_timestamp'] = datetime.now()

# Reorder columns
dim_athlete = dim_athlete[[
    'code', 'gender', 'age',
    'first_year', 'last_year', 'years_participated', 'experience_years',
    'gold_created_timestamp'
]]

dim_athlete.to_csv(os.path.join(GOLD_PATH, "dim_athlete.csv"), index=False)
print(f" Saved dim_athlete: {len(dim_athlete)} rows")
print(f"  Sample codes: {dim_athlete['code'].head(3).tolist()}")
print(f"  Age range: {dim_athlete['age'].min():.0f} - {dim_athlete['age'].max():.0f} years")

Creating dim_athlete from results data
✅ Saved dim_athlete: 7607 rows
  Sample codes: ['001O91NNW62RZP97', '00BKM94J4INLRQAF', '00C4KT4QWQH0RVKP']
  Age range: 0 - 86 years


In [ ]:
# Cell 3: Create dim_sport
print("="*60)
print("Creating dim_sport")
print("="*60)

# Check if sport_clean exists, if not use sport_name
if 'sport_clean' in results_silver.columns:
    sport_col = 'sport_clean'
elif 'sport_name' in results_silver.columns:
    sport_col = 'sport_name'
else:
    raise KeyError("No sport column found in results_silver")

dim_sport = results_silver[[sport_col]].drop_duplicates().copy()
dim_sport = dim_sport.dropna()
dim_sport = dim_sport.reset_index(drop=True)
dim_sport['sport_key'] = range(1, len(dim_sport) + 1)
dim_sport = dim_sport.rename(columns={sport_col: 'sport_name'})
dim_sport = dim_sport[['sport_key', 'sport_name']]
dim_sport['gold_created_timestamp'] = datetime.now()

dim_sport.to_csv(os.path.join(GOLD_PATH, "dim_sport.csv"), index=False)
print(f"Saved dim_sport: {len(dim_sport)} rows")
print(f"  Sports: {dim_sport['sport_name'].tolist()}")

Creating dim_sport
✅ Saved dim_sport: 23 rows
  Sports: ['Athletics', 'Cycling', 'Football', 'Sportgames', 'Gymnastics (Artistic)', 'Aquatics', 'Badminton', 'Table Tennis', 'Bocce', 'Equestrian', 'Adapted Physical Activities', 'Netball', 'Tennis', 'Bowling', 'Motor Activities', 'Floorball', 'Basketball', 'Gymnastics (Rhythmic)', 'Triathlon', 'Judo', 'Sailing', 'Kayaking', 'Swimming']


In [ ]:
# Cell 4: Create dim_region
print("="*60)
print("Creating dim_region")
print("="*60)

dim_region = clubs_silver[['region']].drop_duplicates().copy()
dim_region = dim_region.dropna()
dim_region = dim_region.reset_index(drop=True)
dim_region['region_key'] = range(1, len(dim_region) + 1)
dim_region = dim_region.rename(columns={'region': 'region_name'})
dim_region = dim_region[['region_key', 'region_name']]
dim_region['gold_created_timestamp'] = datetime.now()

dim_region.to_csv(os.path.join(GOLD_PATH, "dim_region.csv"), index=False)
print(f"Saved dim_region: {len(dim_region)} rows")

Creating dim_region
✅ Saved dim_region: 25 rows


In [ ]:
# Cell 5: Create dim_club
print("="*60)
print("Creating dim_club")
print("="*60)

# Include total_participations from the start
dim_club = clubs_silver[['club_id', 'club_name', 'region', 'total_participations']].copy()
print(f"  Starting columns: {dim_club.columns.tolist()}")

# Add region key
dim_club = dim_club.merge(
    dim_region[['region_name', 'region_key']], 
    left_on='region', 
    right_on='region_name', 
    how='left'
)
print(f"  After region merge: {dim_club.columns.tolist()}")


if 'club_name' in results_silver.columns:
    athlete_per_club = results_silver.groupby('club_name')['athlete_id'].nunique().reset_index()
    athlete_per_club = athlete_per_club.rename(columns={'athlete_id': 'certified_athlete_count'})
    print(f"  Calculated athlete count from results_silver")
else:
    # If no club column, create empty count
    athlete_per_club = pd.DataFrame({'club_name': dim_club['club_name'].unique()})
    athlete_per_club['certified_athlete_count'] = 0
    print(f"  WARNING: No club_name in results_silver - setting athlete count to 0")

dim_club = dim_club.merge(athlete_per_club, on='club_name', how='left')
dim_club['certified_athlete_count'] = dim_club['certified_athlete_count'].fillna(0).astype(int)

# Create surrogate key
dim_club['club_key'] = range(1, len(dim_club) + 1)

# Verify all columns exist before selecting
print(f"\n  Final columns available: {dim_club.columns.tolist()}")

# Select final columns (only include columns that exist)
final_cols = ['club_key', 'club_id', 'club_name', 'region_key', 'region', 'certified_athlete_count']
if 'total_participations' in dim_club.columns:
    final_cols.append('total_participations')
else:
    print(f"  WARNING: 'total_participations' not found - adding default value 0")
    dim_club['total_participations'] = 0
    final_cols.append('total_participations')

dim_club = dim_club[final_cols]

dim_club['gold_created_timestamp'] = datetime.now()

dim_club.to_csv(os.path.join(GOLD_PATH, "dim_club.csv"), index=False)
print(f"\nSaved dim_club: {len(dim_club)} rows")
print(f"  Columns saved: {dim_club.columns.tolist()}")

Creating dim_club
  Starting columns: ['club_id', 'club_name', 'region', 'total_participations']
  After region merge: ['club_id', 'club_name', 'region', 'total_participations', 'region_name', 'region_key']
  Calculated athlete count from results_silver

  Final columns available: ['club_id', 'club_name', 'region', 'total_participations', 'region_name', 'region_key', 'certified_athlete_count', 'club_key']

✅ Saved dim_club: 436 rows
  Columns saved: ['club_key', 'club_id', 'club_name', 'region_key', 'region', 'certified_athlete_count', 'total_participations', 'gold_created_timestamp']


In [ ]:
# Cell 6: Create dim_date
print("="*60)
print("Creating dim_date")
print("="*60)

# Get unique years from results
if 'year' in results_silver.columns:
    unique_years = sorted(results_silver['year'].unique())
elif 'source_year' in results_silver.columns:
    unique_years = sorted(results_silver['source_year'].unique())
else:
    # Try to find year column
    year_col = [col for col in results_silver.columns if 'year' in col.lower()]
    unique_years = sorted(results_silver[year_col[0]].unique()) if year_col else [2015, 2016, 2017, 2018, 2019, 2022, 2023, 2024, 2025]

dim_date = pd.DataFrame({'year': unique_years})
dim_date['date_key'] = range(1, len(dim_date) + 1)
dim_date['decade'] = (dim_date['year'] // 10) * 10
dim_date['is_post_covid'] = (dim_date['year'] >= 2022).astype(int)
dim_date['year_name'] = dim_date['year'].astype(str)

dim_date['gold_created_timestamp'] = datetime.now()

dim_date.to_csv(os.path.join(GOLD_PATH, "dim_date.csv"), index=False)
print(f" Saved dim_date: {len(dim_date)} rows")
print(f"  Years: {unique_years}")

Creating dim_date
✅ Saved dim_date: 9 rows
  Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [ ]:
# Cell 7: Create fact_results (using Code from results)
print("="*60)
print("Creating fact_results")
print("="*60)

# Start with results data
fact_results = results_silver[[
    'athlete_id', 'club_name', 'year', 'score', 'rank', 'gender', 'age',
    'is_disqualified', 'rank_missing_flag', 'score_missing_flag', 'sport_name'
]].copy()

# Rename athlete_id to code for consistency
fact_results = fact_results.rename(columns={'athlete_id': 'code'})

# Add sport key
fact_results = fact_results.merge(
    dim_sport[['sport_name', 'sport_key']], 
    on='sport_name', 
    how='left'
)

# Add club key
fact_results = fact_results.merge(
    dim_club[['club_name', 'club_key']], 
    on='club_name', 
    how='left'
)

# Add date key
fact_results = fact_results.merge(
    dim_date[['year', 'date_key']], 
    on='year', 
    how='left'
)

# Calculate rank tier for easier filtering
fact_results['rank_tier'] = pd.cut(fact_results['rank'], 
                                    bins=[0, 1, 3, 10, 1000], 
                                    labels=['Gold (1st)', 'Silver (2-3rd)', 'Bronze (4-10th)', 'Other'],
                                    include_lowest=True)

# Keep only necessary columns
fact_results = fact_results[[
    'code', 'date_key', 'sport_key', 'club_key',
    'gender', 'age', 'rank', 'rank_tier', 'score', 
    'is_disqualified', 'rank_missing_flag', 'score_missing_flag'
]]

fact_results['gold_created_timestamp'] = datetime.now()

fact_results.to_csv(os.path.join(GOLD_PATH, "fact_results.csv"), index=False)
print(f"Saved fact_results: {len(fact_results)} rows")
print(f"  Unique athletes: {fact_results['code'].nunique()}")
print(f"  Years covered: {sorted(dim_date['year'].unique())}")

Creating fact_results
✅ Saved fact_results: 112296 rows
  Unique athletes: 7607
  Years covered: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [ ]:
# Cell 8: Create fact_participation_summary for multi-discipline analysis
print("="*60)
print("Creating fact_participation_summary")
print("="*60)

# Rename for consistency
participation_data = results_silver.rename(columns={'athlete_id': 'code'}).copy()

# Count competitions per athlete per year
competitions_per_year = participation_data.groupby(['code', 'year']).agg({
    'sport_name': 'nunique',
    'rank': lambda x: (x != 999).sum(),
    'is_disqualified': 'sum'
}).reset_index()

competitions_per_year = competitions_per_year.rename(columns={
    'sport_name': 'unique_sports',
    'rank': 'valid_results',
    'is_disqualified': 'disqualifications'
})

competitions_per_year['competition_count'] = participation_data.groupby(['code', 'year']).size().reset_index(name='count')['count'].values
competitions_per_year['multi_sport_flag'] = (competitions_per_year['unique_sports'] > 1).astype(int)

# Add year_key
competitions_per_year = competitions_per_year.merge(
    dim_date[['year', 'date_key']], 
    on='year', 
    how='left'
)

# Athlete lifetime statistics
athlete_lifetime = participation_data.groupby('code').agg({
    'sport_name': 'nunique',
    'year': ['min', 'max', 'nunique'],
    'is_disqualified': 'sum'
}).reset_index()

athlete_lifetime.columns = ['code', 'lifetime_unique_sports', 'first_year', 'last_year', 'years_active', 'total_disqualifications']
athlete_lifetime['career_length'] = athlete_lifetime['last_year'] - athlete_lifetime['first_year'] + 1
athlete_lifetime['sport_changes'] = (athlete_lifetime['lifetime_unique_sports'] > 1).astype(int)

# Merge lifetime stats with yearly data
competitions_per_year = competitions_per_year.merge(athlete_lifetime[['code', 'lifetime_unique_sports', 'sport_changes']], on='code', how='left')

competitions_per_year['gold_created_timestamp'] = datetime.now()

competitions_per_year.to_csv(os.path.join(GOLD_PATH, "fact_participation.csv"), index=False)
print(f"Saved fact_participation: {len(competitions_per_year)} rows")
print(f"  Unique athletes: {competitions_per_year['code'].nunique()}")
print(f"  Athletes with multi-sport: {competitions_per_year['multi_sport_flag'].sum()}")

Creating fact_participation_summary
✅ Saved fact_participation: 27874 rows
  Unique athletes: 7607
  Athletes with multi-sport: 0


In [ ]:
# Cell 9: Validation - Check all gold files
print("="*60)
print("GOLD LAYER VALIDATION")
print("="*60)

gold_files = ['dim_athlete', 'dim_sport', 'dim_region', 'dim_club', 'dim_date', 
              'fact_results', 'fact_participation']

for file in gold_files:
    file_path = os.path.join(GOLD_PATH, f"{file}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"{file}: {len(df)} rows, {len(df.columns)} columns")
    else:
        print(f"{file}: NOT FOUND")



GOLD LAYER VALIDATION
dim_athlete: 7607 rows, 8 columns
dim_sport: 23 rows, 3 columns
dim_region: 25 rows, 3 columns
dim_club: 436 rows, 8 columns
dim_date: 9 rows, 6 columns
fact_results: 112296 rows, 13 columns
fact_participation: 27874 rows, 11 columns

GOLD LAYER COMPLETE - Ready for Power BI
